# The brain's response to a stimulus

This is the runnable version of a chapter of the EEG tutorial. The page, with its references, is at https://crossley.github.io/eeg_tutorial/chapters/04-erps.html

Run the cells from the top, in order (Shift+Enter runs a cell and moves to the next).

In [ ]:
# Run this cell first. It installs MNE and downloads this chapter's data (about 86 MB).
!pip -q install mne

import urllib.request
from pathlib import Path

release = "https://github.com/crossley/eeg_tutorial/releases/download/data-v1/"
files = [
    "sub-01/epochs/sub-01_visit-1_fb-epo.fif",
    "sub-01/epochs/sub-01_visit-1_stim-epo.fif",
]
for file in files:
    target = Path("data/eeg_tutorial_data_v1") / file
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(release + Path(file).name, target)

import mne
mne.set_log_level("WARNING")
mne.viz.set_browser_backend("matplotlib")  # plots appear in the notebook

# MNE's data browser switches off automatic display of later plots; switch it back on before every cell.
import matplotlib.pyplot as plt
get_ipython().events.register("pre_run_cell", lambda *args: plt.ion())
print("Ready.")

The last page ended with an **event-related potential (ERP)**: the
average of many epochs, showing what the brain does each time a
stimulus appears. This page is about why averaging works, how to read
the result, and how to compare two conditions.

From here on we use the whole first visit, about 650 trials, already
cleaned by the lab's pipeline in the way the last page described.

In [ ]:
from pathlib import Path
import mne

data = Path("data/eeg_tutorial_data_v1")
epochs = mne.read_epochs(data / "sub-01/epochs/sub-01_visit-1_stim-epo.fif")
print(len(epochs), "epochs,", len(epochs.ch_names), "channels")
print("from", epochs.tmin, "to", epochs.tmax, "seconds around each stimulus")

The trial table from chapter 2 now travels with the epochs, one row per
epoch:

In [ ]:
epochs.metadata.head()

One housekeeping step. The pipeline marked one electrode as bad for
this visit. We fill it in from its neighbours so that every plot has
all 64 channels.

In [ ]:
print("bad:", epochs.info["bads"])
epochs.interpolate_bads();

## Why we average

Here is the signal at `PO8`, at the back of the head on the right, for
a single trial, and then averaged over the first 10, 100 and 600
trials. The dashed line is the moment the stimulus appeared.

In [ ]:
averages = {"1 trial": epochs[:1].average(),
            "10 trials": epochs[:10].average(),
            "100 trials": epochs[:100].average(),
            "600 trials": epochs[:600].average()}
mne.viz.plot_compare_evokeds(averages, picks="PO8");

In one trial (the blue line) there is nothing to see. The brain's response to the
stimulus is there, but it is a few microvolts, buried in ongoing
activity several times larger that has nothing to do with the stimulus.

That unrelated activity is different on every trial: up on one, down on
the next. The response to the stimulus is nearly the same every time.
So when trials are averaged, the unrelated activity cancels and the
response remains. With 10 trials a shape begins to show;
with 100 it is clear; with 600 the line before the stimulus is almost
perfectly flat.

**The price of a clean average**

The noise shrinks with the *square root* of the number of trials. To
halve the noise you need four times as many trials. This is why EEG
experiments have hundreds of trials per condition, and why you should
be wary of an ERP built from twenty.
**Try it**

Change `"PO8"` to `"Fz"` (front of the head). The response there has a
different shape and is smaller. Does it take more trials before you can
see it?
## Reading an ERP

The average over all epochs, at every electrode at once:

In [ ]:
erp = epochs.average()
erp.plot_joint();

Each line is one electrode; its colour shows where it is on the head
(the small coloured head is the key). The maps along the top show the
whole scalp at the moments marked by the grey lines.

Read it as a sequence in time.

- **Before 0** the lines are flat. Nothing has happened yet, and the
  baseline step set this stretch to zero.
- **Just after 0.1 s** the first sharp peak appears, and the map shows
  it at the back of the head. This is visual cortex registering that
  something has appeared. It takes about a tenth of a second for the
  signal to travel from the eye to the cortex.
- **From about 0.3 s** a slower, broader wave takes over, largest
  between 0.4 and 0.5 s over the back and middle of the head. This later
  activity is less about the
  eyes and more about what the participant does with the stimulus:
  recognising it, deciding, preparing a response.

ERP peaks are named by direction and order or timing: **P1** is the
first positive peak, **N1** the first negative one, **P3** (or P300) a
large positive wave around 0.3 seconds or later.
You will see these names in every ERP paper, and the standard guide to
them is the textbook by Luck (2014).

**Positive and negative are not "more" and "less"**

A positive peak does not mean more brain activity than a negative one.
The sign depends on which way the active patch of cortex happens to be
facing and on the reference. What matters is when a peak
occurs, where on the scalp, and how it changes between conditions.
## Comparing conditions

An ERP on its own says "the brain responded". Experiments are about
differences: does the response change with what was shown, or with
what happened to the participant?

With 64 electrodes and hundreds of time points there is a lot to look
at, and with that many places to look some difference will always
turn up by chance. So it helps to have some idea of where
and when a difference is likely before plotting. The three examples below each start from that.

### Example 1: a difference in the stimulus

The stimuli vary in how wide their stripes are (`x` in the trial
table: low numbers are wide stripes, high numbers are narrow). Neurons
in visual cortex are tuned to stripe width, so wide and
narrow stripes drive different sets of them, and the early visual ERP
is known to change with stripe width.

Visual cortex is at the very back of the head, under `Oz`, and we saw
above that its first response comes at around 0.1 seconds. So that is
a sensible place and time to look.

The trial table lets us pick out any set of epochs by describing it.

In [ ]:
wide = epochs["x < 40"].average()
narrow = epochs["x > 60"].average()
print(wide.nave, "trials with wide stripes,", narrow.nave, "with narrow stripes")

mne.viz.plot_compare_evokeds({"wide": wide, "narrow": narrow}, picks="Oz");

The two lines are together before the stimulus and split apart at 0.1
seconds, right at the first visual response: wide stripes send the voltage up,
narrow stripes send it sharply down. They stay apart until about 0.45
seconds and then come back together. At its largest the split is about
seven microvolts, many times larger than any wobble in the stretch before
time zero.

Was `Oz` the right place? Subtract one average from the other and map
the difference over the whole scalp at that moment:

In [ ]:
difference = mne.combine_evoked([wide, narrow], weights=[1, -1])
difference.plot_topomap(times=[0.105], size=3);

The difference sits at the back of the head and is close to nothing
elsewhere. This is what a clear effect looks like: it is where and when
it makes sense for it to be, and it is large compared with the noise.

### Example 2: a difference in what the participant learns

This one matters more to a lab that studies learning. About a quarter
of a second after people are told they got something wrong, the ERP at
the front and middle of the head is more negative than when they are
told they got it right. This is one of the most studied effects in
EEG, known as the **feedback-related negativity** or,
looking at it the other way up, the **reward positivity**. It is thought to reflect
the brain's signal that an outcome was better or worse than expected,
which is the signal that learning from feedback needs.

For this we need epochs cut around a different event: not the
stimulus, but the moment the feedback appeared. In this task the
feedback was a ring around the stimulus, green for correct and red for
incorrect.

From that literature, the place to look is `FCz`, front and centre,
between about 0.2 and 0.35 seconds after the feedback.

In [ ]:
feedback = mne.read_epochs(data / "sub-01/epochs/sub-01_visit-1_fb-epo.fif")
feedback.interpolate_bads()

correct = feedback["fb == 'Correct'"].average()
incorrect = feedback["fb == 'Incorrect'"].average()
print(correct.nave, "correct trials,", incorrect.nave, "incorrect trials")

mne.viz.plot_compare_evokeds({"correct": correct, "incorrect": incorrect}, picks="FCz");

Time zero is now the moment the ring appeared. At about a quarter of a
second the line for incorrect feedback dips well below the one for
correct feedback, by about four microvolts. That is the
feedback-related negativity, where and when the literature says to
expect it.

Straight afterwards the incorrect line rises above the other. This may
be a later wave like the P3 from earlier on this page, which is often
larger for events that are rarer or more surprising, and
incorrect
feedback was the rarer kind here. That is one plausible reading of one
participant, not a conclusion.

Now map the difference over the scalp, as before:

In [ ]:
difference = mne.combine_evoked([correct, incorrect], weights=[1, -1])
difference.plot_topomap(times=[0.26], size=3);

There is the difference at the front and centre that we were looking
for. But there is also a large one at the very back of the head. What
is that?

Think about what differed between the two kinds of trial. It was not
only the *meaning* of the feedback. One ring was green and the other
was red, and the back of the head is visual cortex. We have just seen,
in Example 1, that a physical difference between two stimuli shows up
there. So the difference at the back is most likely the brain telling
green from red, and has nothing to do with learning.

**Why this matters**

Two conditions almost never differ in only one way. Before saying what
a difference in the brain means, list everything that differed between
the conditions, starting with the dullest: what was on the screen, how
often each condition occurred, which key was pressed. An experiment is
well designed when the dull explanations have been ruled out in
advance.
### Example 3: a difference you should not believe

Back to the epochs around the stimulus. Split them by whether the
participant went on to answer correctly, and look at `Pz`, this time
with no particular reason to expect anything there:

In [ ]:
right = epochs["fb == 'Correct'"].average()
wrong = epochs["fb == 'Incorrect'"].average()

mne.viz.plot_compare_evokeds({"went on to be right": right, "went on to be wrong": wrong}, picks="Pz");

One line runs a little above the other for most of the epoch, by about
one microvolt. It would be easy to tell a story about it.

Look at the stretch before time zero. Nothing can differ there, because
the stimulus has not appeared yet, and still the two lines wobble
apart by more than half a microvolt. That wobble is what noise alone
does to these averages, and the difference after the stimulus is not
much bigger. Compare that with Example 1, where the effect was many
times the size of the wobble.

The stretch before the event is a free look at how big a difference
turns up by chance. Use it every time.

**Try it**

In Example 1, change `picks="Oz"` to `picks="Fz"`, at the front of the
head. Does stripe width make a difference there? Then try comparing the
two categories instead, with `"cat == 'A'"` and `"cat == 'B'"` at `Oz`.
The brain must treat A and B differently, since the participant can
respond to them differently, but the difference is too subtle to see
in an average at one electrode. Chapter 6 shows a more sensitive way
to find it.
## Summary

- A single trial is mostly activity unrelated to the stimulus.
  Averaging many trials cancels it and leaves the response.
- Noise falls with the square root of the number of trials.
- Read an ERP by when its peaks occur and where they are on the scalp.
  Early peaks at the back of the head are visual; later, broader waves
  reflect what is done with the stimulus.
- The sign of a peak is not the amount of activity.
- Science is in the comparison between conditions, and a visible
  difference in one participant is where the work starts.

**In the lab's real pipeline**

ERPs are computed in the same way, per participant and per visit, and
then compared across the five visits to ask how the response changes
as the task is learned. The lab's scripts also build response-locked
epochs, cut around the key press instead of the stimulus, which line up
the activity leading to the response.